In [ ]:
import pandas as pd
import ast
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
### Funzioni generali
def clean_json(json_text):
    list_names = []
    converted_list = ast.literal_eval(json_text)
    for item in converted_list:
        list_names.append(item["name"])
    return list_names

def cleaning_and_preprocessing(df):
    df.dropna(inplace=True)
    df["release_date"] = pd.to_datetime(df["release_date"], errors='coerce')
    df["release_date"] = df["release_date"].dt.strftime('%d/%m/%Y')
    df.rename(columns={"title_x": "title"}, inplace=True)
    df["genres"] = df["genres"].apply(clean_json)
    df["genres"] = df["genres"].apply(lambda x: [genere.lower for genere in x])
    df["keywords"] = df["keywords"].apply(clean_json)
    df["cast"] = df["cast"].apply(clean_json)
    df["crew"] = df["crew"].apply(clean_json)
    df["production_companies"] = df["production_companies"].apply(clean_json)
    df["production_countries"] = df["production_countries"].apply(clean_json)
    df["spoken_languages"] = df["spoken_languages"].apply(clean_json)
    df.drop(["id", "movie_id", "title_y", "budget", "revenue", "production_countries"], axis=1, inplace=True)
    return df

def join_list_elements(dataframe, columns):
    for col in columns:
        dataframe[col] = dataframe[col].apply(lambda x: " ".join(x))
    return dataframe

def find_genres(user_query, unique_genres):
    query_lowered = user_query.lower()
    genres_found = [gen for gen in unique_genres if gen in query_lowered]
    return genres_found

In [ ]:
### Preprocessing e Pulizia
movies = pd.read_csv("Dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("Dataset/tmdb_5000_credits.csv")
df = pd.merge(movies, credits, left_on="id", right_on="movie_id")

df = cleaning_and_preprocessing(df)
df.to_csv('tmdb_5000_pulito.csv', index=False)
print("Dataset salvato con successo!")

In [ ]:
### Testo unico per la Sentence
df = join_list_elements(df, ["cast", "crew", "keywords", "production_companies"])

### Lista Generi
all_genres = [genres for sublist in df["genres"] for genres in sublist]
unique_genres = sorted(list(set(all_genres)))
#print(f"Generi individuati:{unique_genres}")


### Sentence Embedding e Cosine Similarity
sentence_var = (df["overview"] + " " + df["cast"] + " " + df["crew"] + " " + df["keywords"] + " " + df["tagline"] +" " + df["production_companies"]) 
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_sentence = model.encode(sentence_var.tolist())
#similarity_matrix = cosine_similarity(embeddings_sentence)

In [ ]:
 ## Calcolo peso ed embedding della query
def calc_sentence_weight(user_query):
    query_embedding = model.encode([user_query], convert_to_numpy=True)
    sentence_score = cosine_similarity(query_embedding, embeddings_sentence.numpy())[0]
    return sentence_score


## Findings dei generi presenti nella query
def find_genres(user_query, unique_genres):
    query_lowered = user_query.lower()
    genres_found = [gen for gen in unique_genres if gen in query_lowered]
    return genres_found


## Confronto tra i generi nella query e i film per genere
def calc_genres_weight(query_genres):
    genres_score = []
    for genres in df["genres"]:
        if len(query_genres) == 0:
            genres_score.append(0)
        else:
            intersection = set(query_genres) & set(genres)
            num_genres = len(intersection) / len (query_genres)
            genres_score.append(num_genres)
    return genres_score

## Calcolo score totale con w
def calc_total_score(sentence_score, genres_score):
    total_score = (np.array(sentence_score) * 0.7) + (np.array(genres_score) * 0.3)
    return total_score


def search_film():
    user_query = input("Inserisci la tua query: ")
    # Encoding della query e calcolo del peso
    sentence_score = calc_sentence_weight(user_query)
    # Findings dei generi nella query e calcolo del peso
    query_genres = find_genres(user_query, unique_genres)
    genres_score = calc_genres_weight(query_genres)
    total_score = calc_total_score(sentence_score, genres_score)
    results = list(enumerate(total_score))
    top_10 = sorted(results, key=lambda x: x[1], reverse=True)[:10]
    index = [i[0] for i in top_10]
    table = df.iloc[index][["title", "release_date", "genres", "spoken_languages", "homepage"]]
    print("\nFilm Consigliati per te")
    print(table.to_string(index=False))
    return table 


search_film()